# Setup

In [1]:
from pathlib import Path
import sys

import numpy as np

# Ensure project root is on sys.path even if notebook cwd is not repo root.
cwd = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "src").exists() and (candidate / "config.yaml").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root (expected src/ and config.yaml).")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_config, setup_logging
from src.dataset_io import scan_dataset, load_sample
from src.visualization import plot_activation_topk, animate_motion_interactive, show_dash_app, show_dash_smplx_motion
from src.smplx_to_opensim import smplx_to_mot
from src.notebook_helpers import (
    build_original_motion_doll_animation,
    format_sequence_description,
    run_pipeline_with_progress,
    summarize_solver_metrics,
    write_solver_metrics_artifact,
)

CONFIG_PATH = (PROJECT_ROOT / "config.yaml").resolve()
cfg = load_config(CONFIG_PATH)
setup_logging("INFO")

paths = cfg["paths"]
config_dir = CONFIG_PATH.parent

# Resolve config-relative paths to absolute paths for stable notebook runs.
motionx_root = (config_dir / paths["motionx_root"]).resolve()
output_root = (config_dir / paths["output_root"]).resolve()
tmp_root = (config_dir / paths["temp_dir"]).resolve()

print(f"Notebook Python: {sys.executable}")
print(f"Project root: {PROJECT_ROOT}")
print(f"OpenSim Python (config): {paths.get('opensim_python_exe')}")

try:
    import opensim  # type: ignore
    OPENSIM_AVAILABLE = True
    print(f"OpenSim import OK: {getattr(opensim, '__version__', 'unknown')}")
except ImportError:
    OPENSIM_AVAILABLE = False
    print("OpenSim import FAILED in current notebook kernel.")
    print(
        "→ conda activate musclemap-data  # from environment.yml\n"
        "→ pip install -e .   # editable src package\n"
        "→ python scripts/setup_check.py --config config.yaml"
    )

Notebook Python: /opt/anaconda3/envs/musclemap-data/bin/python
Project root: /Users/maksimpecin/Library/CloudStorage/OneDrive-Личная/Thesis/code/musclemap-data
OpenSim Python (config): /opt/anaconda3/envs/musclemap-data/bin/python
OpenSim import OK: 4.6


In [2]:
from src.visualization import coords_to_skeleton_joints
import numpy as np

# 45° knee flexion — shin should move POSTERIOR (negative Z in world)
frame = {"knee_angle_r": np.radians(45), "pelvis_ty": 0.9}
j = coords_to_skeleton_joints(frame)
r_knee, r_ankle = j[4], j[7]

dz = float(r_ankle[2] - r_knee[2])   # world Z: positive = forward, negative = back
print(f"Ankle Z relative to knee: {dz:.3f}m  (should be negative = posterior)")
assert dz < -0.05, f"Knee bends forward instead of backward! dz={dz:.3f}"
print("PASSED: knee flexion moves shin posteriorly")

# 30° hip flexion — thigh should swing ANTERIOR (positive Z)
frame2 = {"hip_flexion_r": np.radians(30), "pelvis_ty": 0.9}
j2 = coords_to_skeleton_joints(frame2)
r_hip, r_knee2 = j2[1], j2[4]
dz2 = float(r_knee2[2] - r_hip[2])
print(f"Knee Z relative to hip: {dz2:.3f}m  (should be positive = anterior)")
assert dz2 > 0.05, f"Hip flexion moves thigh posterior instead of anterior! dz={dz2:.3f}"
print("PASSED: hip flexion swings thigh anteriorly")

Ankle Z relative to knee: -0.283m  (should be negative = posterior)
PASSED: knee flexion moves shin posteriorly
Knee Z relative to hip: 0.200m  (should be positive = anterior)
PASSED: hip flexion swings thigh anteriorly


# Dataset Size (for SAMPLE_IDX bounds)

In [3]:
samples = scan_dataset(motionx_root, cfg)
if not samples:
    raise RuntimeError("No samples found — check paths.motionx_root and dataset layout.")

dataset_size = len(samples)
print(f"Dataset size: {dataset_size}")
print(f"Valid SAMPLE_IDX range: 0..{max(0, dataset_size - 1)}")

Dataset size: 12025
Valid SAMPLE_IDX range: 0..12024


# Select Sequence

In [4]:
import pandas as pd
from IPython.display import HTML, display


SEARCH_WORD = "walk"  # Easy to change.



rows = [
    {
        "SAMPLE_IDX": i,
        "name": s.id,
        "length in frames": int(np.load(s.motion_path, mmap_mode="r").shape[0]),
    }
    for i, s in enumerate(samples)
    if SEARCH_WORD in s.id.lower()
]

table = pd.DataFrame(rows).sort_values("length in frames", ascending=True)

try:
    display(table.style.set_table_attributes(
        'style="display:block; max-height:420px; overflow-y:auto; overflow-x:auto;"'
    ))
except Exception:
    display(HTML(f'<div style="max-height:420px; overflow:auto;">{table.to_html(index=False)}</div>'))

,SAMPLE_IDX,name,length in frames
2288,7330,motion_generation/smplx_322/idea400/Wiping_with_a_rag_during_walking_clip1,6
2960,11254,motion_generation/smplx_322/idea400/walking_during_Giving_Objects_clip3,6
3705,11999,motion_generation/smplx_322/idea400/walking_while_Wiping_with_a_rag_1_clip4,6
515,1697,motion_generation/smplx_322/idea400/Holding_things_in_your_hands_and_walking_at_the_same_time_1_clip3,6
2710,11004,motion_generation/smplx_322/idea400/walking_and_Signing_a_contract_at_the_same_time_clip2,6
3179,11473,motion_generation/smplx_322/idea400/walking_during_Taking_pictures_clip1,6
3180,11474,motion_generation/smplx_322/idea400/walking_during_Taking_pictures_clip4,6
1398,4593,motion_generation/smplx_322/idea400/Simultaneously_Side_Kick_and_walking_2_clip1,6
3642,11936,motion_generation/smplx_322/idea400/walking_while_Spread_your_hands_2_clip2,6
3008,11302,motion_generation/smplx_322/idea400/walking_during_Kicking_a_soccer_ball_1_clip1,6


In [5]:
SAMPLE_IDX = 6186 # TUNE THIS


sample = samples[SAMPLE_IDX]
data = load_sample(sample)
motion = data["motion"]
sample.id, motion.shape

('motion_generation/smplx_322/idea400/Simultaneously_walking_and_Kicking_a_soccer_ball_1_clip1',
 (145, 322))

# Raw Motion — Joint Angles Before/After Filter

In [6]:
from src.smplx_to_opensim import butterworth_filter
from src.utils import joint_velocity_clamp as jvc

conv = cfg["conversion"]
ds = cfg["dataset"]
fps = float(ds["fps"])
body = motion[:, 3:66]
stacked = body.astype(np.float64)
filt = butterworth_filter(
    stacked,
    int(conv["filter_order"]),
    float(conv["filter_cutoff_hz"]),
    fps,
)
clamped = jvc(filt.astype(np.float32), float(conv["max_joint_velocity_rad_s"]), fps)
body.shape, filt.shape, clamped.shape

((145, 63), (145, 63), (145, 63))

# Original SMPL-X Motion Visualization

In [7]:
show_dash_smplx_motion(
    smplx_motion=motion,
    config=cfg,
    inline=True,
)

2026-05-04 23:16:55,041 src.smplx_joint_regressor INFO No SMPL-X regressor path configured; using geometric fallback.
2026-05-04 23:16:55,465 src.dash_app INFO Dash app prepared with 145 frames and 1 muscles.
2026-05-04 23:16:55,465 src.visualization WARNING jupyter_dash not installed; falling back to browser tab. Install with: pip install jupyter-dash


# Sequence-Level Text Description

In [8]:
report = format_sequence_description(sample.id, str(data.get("semantic", "")))
print(report)

Sequence ID: motion_generation/smplx_322/idea400/Simultaneously_walking_and_Kicking_a_soccer_ball_1_clip1

Sequence-level description:
1.3805358409881592 -0.8334254622459412 -0.9363736510276794 -0.3003528118133545 -0.020363159477710724 0.07985253632068634 0.1345561295747757 0.017657672986388206 0.12922334671020508 0.22662442922592163 0.06648080050945282 0.056665271520614624 0.35714191198349 -0.05568165332078934 0.13699419796466827 0.05238160490989685 0.006960710976272821 -0.042303960770368576 -0.09234094619750977 -0.0020123692229390144 -0.04571353644132614 -0.10754484683275223 0.15999993681907654 -0.058233123272657394 -0.11696028709411621 -0.1677841693162918 0.03528672829270363 0.23296721279621124 0.01627993769943714 0.009155919775366783 -0.03091477043926716 -0.002377082360908389 0.03527229279279709 -0.024743979796767235 0.0014766486128792167 -0.03547709062695503 0.36701127886772156 -0.023858370259404182 0.062245432287454605 0.012481136247515678 0.15410424768924713 -0.06256824731826782

# Run OpenSim Pipeline

In [ ]:
import os
import sys
import tempfile
from contextlib import contextmanager

DRY_RUN = False  # TUNE THIS — set True to skip OpenSim
SAVE_SOLVER_METRICS = False  # TUNE THIS — set True to persist metrics JSON

seq_tmp = (tmp_root / sample.id).resolve()
seq_tmp.mkdir(parents=True, exist_ok=True)

if not DRY_RUN and not OPENSIM_AVAILABLE:
    _py = paths.get("opensim_python_exe", "<run scripts/setup_check.py>")
    raise RuntimeError(
        "OpenSim is not importable in this notebook kernel.\n\n"
        f"Current Python: {sys.executable}\n"
        f"Config OpenSim Python: {_py}\n\n"
        "Use the conda env from environment.yml (same Python should match config after setup_check):\n"
        "  conda activate musclemap-data\n"
        f'  python -m pip install -e "{PROJECT_ROOT}"\n'
        "  python -m ipykernel install --user --name musclemap-data --display-name \"musclemap-data\"\n"
        "Then pick the musclemap-data kernel and restart.\n\n"
        "Option — set DRY_RUN = True in this cell to skip OpenSim (no static optimization).\n"
    )

@contextmanager
def _capture_native_output():
    """Capture noisy native OpenSim stdout/stderr to a file for filtering."""
    with tempfile.NamedTemporaryFile(mode="w+", delete=False) as tmp:
        cap_path = tmp.name
    saved_out = os.dup(1)
    saved_err = os.dup(2)
    f = open(cap_path, "w")
    try:
        os.dup2(f.fileno(), 1)
        os.dup2(f.fileno(), 2)
        yield cap_path
    finally:
        f.flush()
        f.close()
        os.dup2(saved_out, 1)
        os.dup2(saved_err, 2)
        os.close(saved_out)
        os.close(saved_err)

try:
    with _capture_native_output() as _cap_path:
        acts, muscle_names, so_metrics = run_pipeline_with_progress(
            sample.motion_path,
            cfg,
            seq_tmp,
            dry_run=DRY_RUN,
            save_solver_log=False,
            save_solver_metrics=SAVE_SOLVER_METRICS,
            expected_frames=motion.shape[0],
        )
except RuntimeError as exc:
    cause = exc.__cause__
    cause_msg = str(cause) if cause else ""
    if "OpenSim not found" in str(exc) or "OpenSim not found" in cause_msg:
        _py = paths.get("opensim_python_exe", "")
        raise RuntimeError(
            "OpenSim import failed in-process (wrong kernel?). "
            f'Switch to kernel running "{_py}" or register ipykernel there; '
            "see the RuntimeError text when OPENSIM_AVAILABLE is False, or set DRY_RUN=True."
        ) from exc
    raise

# Show useful information, hide warning spam.
in_model_block = False
with open(_cap_path, "r", encoding="utf-8", errors="ignore") as _f:
    for _ln in _f:
        s = _ln.rstrip("\n")
        low = s.lower()
        if "[warning]" in low:
            continue
        if "model:" in s and "[" not in s:
            in_model_block = True
            print(s)
            continue
        if in_model_block:
            if not s.strip():
                in_model_block = False
                continue
            print(s)
            continue
        if "[info]" in low or " info " in low or low.startswith("info"):
            print(s)

os.remove(_cap_path)

acts.dtype, acts.shape, len(muscle_names), len(so_metrics)


SO frames:   0%|          | 0/145 [00:00<?, ?frame/s]

2026-05-04 23:16:55,527 src.smplx_joint_regressor INFO No SMPL-X regressor path configured; using geometric fallback.


In [ ]:
# Static Optimization metrics summary (parsed from solver output)

so_summary = summarize_solver_metrics(so_metrics)
so_summary

if SAVE_SOLVER_METRICS:
    metrics_path = (seq_tmp / "notebook_so_metrics.json").resolve()
    write_solver_metrics_artifact(metrics_path, so_metrics)
    print(f"Saved solver metrics: {metrics_path}")

if so_metrics:
    so_worst = sorted(so_metrics, key=lambda x: x["constraint_violation"], reverse=True)[:5]
    so_worst

# Muscle Activations

In [ ]:
import matplotlib.pyplot as plt

fig_ts, ax = plt.subplots(figsize=(10, 3))
ax.plot(acts[:, : min(5, acts.shape[1])])
ax.set_title("First muscles (subset)")
ax.set_xlabel("Frame")
ax.set_ylabel("Activation")
fig_ts

# Top-K Activation Variance — `K = 10  # TUNE THIS`

In [ ]:
K = 10  # TUNE THIS

fig_topk = plot_activation_topk(acts, muscle_names, K)
fig_topk

# Interactive Animation

In [ ]:
# The .mot file was already written by the pipeline step (Cell 14).
# Re-use it directly so the animation is driven by the same coordinates
# that were fed into Static Optimisation - no separate smplx_to_mot call needed.
_mot_path = (tmp_root / sample.id / "viz_coords.mot").resolve()

# Fallback: if the pipeline used a different temp path, regenerate the .mot.
if not _mot_path.exists():
    _seq_tmp = (tmp_root / sample.id).resolve()
    _seq_tmp.mkdir(parents=True, exist_ok=True)
    smplx_to_mot(motion, cfg, _mot_path)

show_dash_app(
    mot_path=_mot_path,
    activations=acts,
    muscle_names=muscle_names,
    config=cfg,
    smplx_motion=motion,
    inline=True,
)